<a href="https://colab.research.google.com/github/16howarthm/pgh-road-ai/blob/main/category_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pittsburgh street-closure category analysis
Reproducible experiment with six balanced categories, fixed prompt-development and held-out splits, and exact 0/1/several-example prompt conditions.

In [ ]:
import importlib
import json
import os
import shutil
import subprocess
import sys
import pandas as pd
from pathlib import Path

os.chdir("/content")

REPOSITORY_URL = "https://github.com/16howarthm/pgh-road-ai.git"
REPOSITORY_DIR = Path("/content/pgh-road-ai")

from google.colab import userdata

if REPOSITORY_DIR.exists():
    shutil.rmtree(REPOSITORY_DIR)
subprocess.run(
    [
        "git", "clone", "--branch", "main", "--single-branch",
        REPOSITORY_URL, str(REPOSITORY_DIR),
    ],
    check=True,
)

os.chdir(REPOSITORY_DIR)
subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "--quiet",
        "pandas>=2.0", "scikit-learn>=1.3", "openai>=1.0",
    ],
    check=True,
)

openrouter_api_key = userdata.get("openrouter_api_key")
if not openrouter_api_key:
    raise RuntimeError(
        "openrouter_api_key is unavailable. Add it to Colab Secrets, grant notebook access, "
        "and rerun this cell."
    )
os.environ["openrouter_api_key"] = openrouter_api_key

importlib.invalidate_caches()
sys.modules.pop("road_ai", None)
sys.path.insert(0, str(REPOSITORY_DIR))
road_ai = importlib.import_module("road_ai")
required_road_ai_names = [
    "DEFAULT_CATEGORIES",
    "balanced_sample",
    "load_fixed_splits",
    "make_examples",
    "run_experiment",
    "save_fixed_splits",
    "score_results",
    "split_prompt_test",
    "validate_splits",
]
missing_road_ai_names = [
    name for name in required_road_ai_names if not hasattr(road_ai, name)
]
if missing_road_ai_names:
    raise RuntimeError(
        "Fresh clone road_ai.py is missing required names: "
        + ", ".join(missing_road_ai_names)
    )
from road_ai import (
    DEFAULT_CATEGORIES,
    balanced_sample,
    load_fixed_splits,
    make_examples,
    run_experiment,
    save_fixed_splits,
    score_results,
    split_prompt_test,
    validate_splits,
)

PROJECT_ROOT = REPOSITORY_DIR
DATA_PATH = PROJECT_ROOT / "pgh data.csv"
SPLIT_DIR = PROJECT_ROOT / "category_splits"
OUTPUT_DIR = PROJECT_ROOT / "category_results"
LABEL_COL = "work_type"
CATEGORIES = DEFAULT_CATEGORIES
MODELS = ["meta-llama/llama-4-scout", "openai/gpt-oss-120b"]
PROMPT_CONDITIONS = {"zero": 0, "one": 1, "several": 6}
RANDOM_STATE = 42

print("Project root:", PROJECT_ROOT)
print("OpenRouter key available:", bool(openrouter_api_key))

## Load or create the fixed splits
The first run de-duplicates permits, reports and removes conflicting-label description groups, samples 50 unique descriptions per category, and saves fixed 120/180 splits. Later runs load those same records.

In [ ]:
prompt_path = SPLIT_DIR / "prompt_development.csv"
test_path = SPLIT_DIR / "heldout_test.csv"
audit_path = SPLIT_DIR / "split_audit.json"

if prompt_path.exists() and test_path.exists():
    prompt_dev, test = load_fixed_splits(LABEL_COL, SPLIT_DIR)
    audit = json.loads(audit_path.read_text()) if audit_path.exists() else {}
else:
    df = pd.read_csv(DATA_PATH)
    sample, audit = balanced_sample(
        df, LABEL_COL, CATEGORIES, n_per_category=50,
        random_state=RANDOM_STATE, return_audit=True,
    )
    prompt_dev, test = split_prompt_test(
        sample, LABEL_COL, prompt_n=120, test_n=180, random_state=RANDOM_STATE
    )
    save_fixed_splits(prompt_dev, test, LABEL_COL, SPLIT_DIR, audit)

validate_splits(prompt_dev, test, LABEL_COL)
print(audit)
print("Prompt-development counts:\n", prompt_dev[LABEL_COL].value_counts().sort_index())
print("Held-out counts:\n", test[LABEL_COL].value_counts().sort_index())

## Run every model and prompt condition
This fresh run executes all 1,080 requests using the same saved 180-record held-out set. `one` contains exactly one labeled example total; `several` contains six total (one per category).

In [ ]:
if not openrouter_api_key:
    raise RuntimeError(
        "openrouter_api_key is unavailable. Add it to Colab Secrets, grant notebook access, "
        "and rerun the setup cell."
    )

all_results = []
for model in MODELS:
    for condition, n_examples in PROMPT_CONDITIONS.items():
        examples = make_examples(
            prompt_dev, LABEL_COL, n_examples=n_examples, random_state=RANDOM_STATE
        )
        assert len(examples) == n_examples
        run = run_experiment(
            test, LABEL_COL, model=model, categories=CATEGORIES, examples=examples
        )
        assert len(run) == 180
        assert run.permit_id.tolist() == test.permit_id.tolist()
        run["model"] = model
        run["prompt_condition"] = condition
        run["n_examples"] = n_examples
        all_results.append(run)

results = pd.concat(all_results, ignore_index=True)
combination_sizes = results.groupby(["model", "prompt_condition"]).size()
assert len(results) == 1080
assert len(combination_sizes) == 6
assert combination_sizes.eq(180).all()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results_path = OUTPUT_DIR / "classification_results.csv"
results.to_csv(results_path, index=False)
assert len(pd.read_csv(results_path)) == 1080
print("Raw results saved to:", results_path)
results.head()

## Metrics and confusion matrices
Failures and invalid responses remain in the 180-row denominator and appear in the `__FAILED__` confusion-matrix column.

In [ ]:
summary_rows = []
matrix_rows = []
for (model, condition), group in results.groupby(["model", "prompt_condition"], sort=False):
    scores = score_results(group, CATEGORIES)
    assert scores["n_scored"] == 180
    summary_rows.append({
        "model": model,
        "prompt_condition": condition,
        "n_examples": int(group.n_examples.iloc[0]),
        "accuracy": scores["accuracy"],
        "macro_f1": scores["macro_f1"],
        "failed_predictions": scores["n_failures"],
        "prompt_tokens": group.prompt_tokens.sum(),
        "completion_tokens": group.completion_tokens.sum(),
    })
    matrix = scores["confusion_matrix"]
    display(model, condition, matrix)
    for actual, row in matrix.iterrows():
        for predicted, count in row.items():
            matrix_rows.append({
                "model": model, "prompt_condition": condition,
                "actual": actual, "predicted": predicted, "count": int(count),
            })

summary = pd.DataFrame(summary_rows)
confusion_matrices = pd.DataFrame(matrix_rows)
summary_path = OUTPUT_DIR / "classification_summary.csv"
confusion_path = OUTPUT_DIR / "confusion_matrices.csv"
summary.to_csv(summary_path, index=False)
confusion_matrices.to_csv(confusion_path, index=False)
assert len(summary) == 6
assert results_path.exists() and summary_path.exists() and confusion_path.exists()
print("All outputs saved under:", OUTPUT_DIR)
display(summary)

## Error review

In [ ]:
errors = results[results.actual.ne(results.predicted)].sort_values(
    ["model", "prompt_condition", "confidence"], na_position="first"
)
display(errors[[
    "model", "prompt_condition", "permit_id", "description",
    "actual", "predicted", "confidence", "reason", "error",
]].head(50))